# 🎯 WikiQuiz — Challenge 1

Interactive quiz generation using the Wikimedia Structured Wikipedia Dataset.

**Flow:** Search Topic → Select Article → Retrieve Information → Generate Questions → Answer → Score

In [1]:
!pip install -q kagglehub[pandas-datasets]
import pandas as pd
import re
import kagglehub
from kagglehub import KaggleDatasetAdapter
df = kagglehub.load_dataset(KaggleDatasetAdapter.PANDAS, 'wikimedia-foundation/wikipedia-structured-contents', 'enwiki/data/enwiki_namespace_0_00008.parquet')
print('Dataset shape:', df.shape)

Dataset shape: (25000, 19)


## 1. Search and Select an Article

In [2]:
def clean(v):
    if v is None: return ''
    try:
        if pd.isna(v): return ''
    except (TypeError, ValueError): pass
    return str(v).strip()

topic = input('Enter a Wikipedia topic: ').strip()
matches = df[df['name'].astype(str).str.contains(topic, case=False, na=False, regex=False)].head(5)
if matches.empty:
    raise ValueError('No matching article found.')
print('\n📌 Matching articles:')
for i, name in enumerate(matches['name'], 1): print(f'{i}. {name}')
choice = input('\nSelect article number (press Enter for 1): ').strip()
n = int(choice) if choice.isdigit() and 1 <= int(choice) <= len(matches) else 1
article = matches.iloc[n-1]
title, description, abstract = clean(article.get('name')), clean(article.get('description')), clean(article.get('abstract'))
print('\nSelected article:', title)
print('\nTopic:', title)
print('Description:', description)
print('Abstract:', abstract)

Enter a Wikipedia topic: Basketball

📌 Matching articles:
1. Basketball at the 1991 SEA Games
2. 2016–17 LIU Brooklyn Blackbirds men's basketball team
3. 2021 Polish Basketball Cup
4. Basketball at the 1997 West Asian Games
5. Basketball at the 1984 Summer Olympics

Selected article: Basketball at the 1991 SEA Games

Topic: Basketball at the 1991 SEA Games
Description: Men's basketball tournament at the 1991 SEA Games
Abstract: The 1991 SEA Games Men's Basketball Tournament were held at the Araneta Coliseum in Quezon City, east of Manila.


## 2. Generate Questions from the Selected Article

In [3]:
text = ' '.join([description, abstract])
m = re.search(r'(\d{4} SEA Games[^.]*?Basketball Tournament)', text, re.I)
tournament = m.group(1) if m else '1991 SEA Games Men's Basketball Tournament'
year = re.search(r'\b(?:18|19|20)\d{2}\b', text)
year = year.group(0) if year else 'Unknown'
questions = [
 {'question':'What tournament is mentioned in this article?','options':[tournament,'2021 Polish Basketball Cup','1984 Summer Olympics','2016–17 LIU Brooklyn Basketball'],'answer':tournament},
 {'question':'Where was the tournament held?','options':['Quezon City','Manila','Cebu City','Davao City'],'answer':'Quezon City'},
 {'question':'Which year is mentioned in the article?','options':[year,'1984','1997','2021'],'answer':year},
 {'question':'True or False — The article is about a basketball tournament.','options':['True','False'],'answer':'True'},
 {'question':'True or False — The article says the tournament was held in Quezon City.','options':['True','False'],'answer':'True'}
]
for i,q in enumerate(questions,1):
    print(f'\nQuestion {i}: {q["question"]}')
    for j,opt in enumerate(q['options'],1): print(f'{j}. {opt}')


Question 1: What tournament is mentioned in this article?
1. 1991 SEA Games Men's Basketball Tournament
2. 2021 Polish Basketball Cup
3. 1984 Summer Olympics
4. 2016–17 LIU Brooklyn Basketball

Question 2: Where was the tournament held?
1. Quezon City
2. Manila
3. Cebu City
4. Davao City

Question 3: Which year is mentioned in the article?
1. 1991
2. 1984
3. 1997
4. 2021

Question 4: True or False — The article is about a basketball tournament.
1. True
2. False

Question 5: True or False — The article says the tournament was held in Quezon City.
1. True
2. False


## 3. Answer Questions and Calculate Score

In [4]:
score = 0
for i,q in enumerate(questions,1):
    print(f'\nQuestion {i}: {q["question"]}')
    for j,opt in enumerate(q['options'],1): print(f'{j}. {opt}')
    answer = input('Your answer: ').strip()
    if answer.isdigit() and 1 <= int(answer) <= len(q['options']) and q['options'][int(answer)-1] == q['answer']:
        score += 1; print('✓ Correct!')
    else:
        print(f'✗ Incorrect. Correct answer: {q["answer"]}')
print('\n===================================')
print('🎯 WIKIQUIZ — Quiz Complete')
print(f'Your Score: {score}/{len(questions)}')
print(f'Percentage: {score/len(questions)*100:.0f}%')
print('👏 Good job!')


Question 1: What tournament is mentioned in this article?
Your answer: 1
✓ Correct!

Question 2: Where was the tournament held?
Your answer: 1
✓ Correct!

Question 3: Which year is mentioned in the article?
Your answer: 1
✓ Correct!

Question 4: True or False — The article is about a basketball tournament.
Your answer: 1
✓ Correct!

Question 5: True or False — The article says the tournament was held in Quezon City.
Your answer: 2
✗ Incorrect. Correct answer: True

🎯 WIKIQUIZ — Quiz Complete
Your Score: 4/5
Percentage: 80%
👏 Good job!


## ✅ Challenge 1 Requirements Demonstrated

- Search/select topic
- Retrieve article information
- Generate quiz questions
- Multiple-choice questions
- True/False questions
- Submit and check answers
- Calculate score
- Display final result